# Ranking estadístico — ¿qué criptomoneda es más eficiente?

Este notebook responde la pregunta central del proyecto: dadas las cinco criptomonedas, ¿cuál ofrece el mejor retorno por unidad de riesgo asumida?

Se recalculan las métricas sobre el dataset completo (en lugar de leer `metricas_criptos.csv`) para mantener este notebook autocontenido. Un cambio en el notebook 02 no invalidaría silenciosamente los resultados aquí.

In [ ]:
# 04_recomendaciones_estadisticas.ipynb
# Recomendaciones estadísticas para compra de criptomonedas
# Basado en retornos, volatilidad, drawdown y precio mínimo histórico

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

## 1. Carga del dataset unificado

In [ ]:
# 1. Cargar dataset unificado

## 2. Retorno diario y drawdown

Misma lógica que en el notebook 02. Se agrupa por cripto antes de `pct_change()` para evitar retornos ficticios en la primera fila de cada activo.

In [ ]:
ruta_csv = "../datos/procesados/precios_diarios.csv"
df = pd.read_csv(ruta_csv, encoding="utf-8", sep=';')
df["fecha"] = pd.to_datetime(df["fecha"], dayfirst=True)

# Ordenar por cripto y fecha
df = df.sort_values(["cripto_id", "fecha"]).reset_index(drop=True)

## 3. Drawdown acumulado

In [ ]:
# 2. Calcular retornos diarios

## 4. Ranking por ratio riesgo/retorno

Se ordena descendente por ratio riesgo/retorno. **El activo con mayor retorno promedio (ZEN, 41.2%) queda en último lugar** porque su volatilidad es 85 veces superior a la de BTC. Usar solo el retorno para seleccionar llevaría a elegir el activo más arriesgado del conjunto.

El precio mínimo histórico y su fecha se incluyen como referencia de valoración relativa, no como señal de compra futura.

In [ ]:
df["retorno_diario"] = df.groupby("cripto_id")["cierre"].pct_change()

## 5. Conclusión del ranking

Se imprime la criptomoneda con mejor ratio y su precio mínimo histórico. La interpretación es estrictamente retrospectiva: indica qué activo fue históricamente más eficiente, no cuál lo será.

**Limitaciones:**
- El ratio no descuenta ninguna tasa libre de riesgo, por lo que no es directamente comparable con un ratio de Sharpe estándar.
- No se considera liquidez ni volumen: un activo con buen ratio puede ser difícil de comprar en cantidad.
- Los mercados de criptomonedas pueden cambiar de régimen de volatilidad abruptamente por eventos regulatorios o de adopción.

In [ ]:
# 3. Calcular drawdown diario

## 6. Visualización

El precio mínimo histórico se marca sobre la serie completa de la criptomoneda recomendada. El punto rojo indica el momento de menor precio en el histórico disponible, no el momento óptimo de entrada en el futuro.

In [ ]:
df["cierre_max"] = df.groupby("cripto_id")["cierre"].cummax()
df["drawdown"] = df["cierre_max"] - df["cierre"]

## 7. Persistencia del ranking

In [ ]:
# 4. Estadísticas agregadas por cripto